# Browsing Automation

## Resources:

- https://playwright.dev/python/docs/api/class-page 

## Test URLs
### Works on PlayWright: 
- https://jobs.workable.com/view/5Q8tVt6rfpWg4A8H8FkFiB/remote-ai-solutions-engineer-in-buenos-aires-at-hire-overseas

- https://webscraper.io/test-sites/e-commerce/allinone

- https://jobs.generalcatalyst.com/companies/sonarsource/jobs/74080968-ai-engineer 


### Not Work on PlayWright: 
- https://wellfound.com/jobs/4050171-senior-ai-engineer-agentic-devsecops-automation-j318 

In [2]:
from typing import Optional, List
from dataclasses import dataclass

@dataclass
class Job:
    title: str
    url: str
    text: Optional[str]
    searched_via: str
    low_quality: bool = False

In [3]:
from playwright.sync_api import sync_playwright

## Figure out a way to minimalizing noise of HTML tags the best 
Before passing to the LLM

In [32]:
from playwright.async_api import async_playwright
import asyncio
import re

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        await page.goto("https://jobs.generalcatalyst.com/companies/sonarsource/jobs/74080968-ai-engineer")
        
        desc_patterns = ["Description", "About the Role", "Job Description", "What You’ll Be Doing", "Why should I Apply"]

        for desc_pattern in desc_patterns:
            locator = page.get_by_role("heading", name=re.compile(desc_pattern, re.I))


            if await locator.count() > 0: 
                section = locator.first.locator("xpath=..")
                text = await section.inner_text()

                print(f"FOUND SECTION: {desc_pattern.upper()}")
                print(text)
                break
        
        await browser.close()

await run()

## Trials and Errors: 


### page.get_by_text
Inconsistent because it’s random of the location where the first selected text matches. It can’t pull out all the text stably all the time. 

In [27]:
from playwright.async_api import async_playwright
import asyncio
import re

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        await page.goto("https://www.builtinnyc.com/job/ai-engineer/8349500")
        
        desc_patterns = ["Description", "About the Role", "Job Description", "What You’ll Be Doing", "skills"]

        for desc_pattern in desc_patterns:
            locator = page.get_by_text(desc_pattern.lower(), exact=False)


            if await locator.count() > 0: 
                # go one level up in the DOM (parent node) with xpath
                section = locator.first.locator("xpath=..")
                text = await section.inner_text()

                print(f"FOUND SECTION: {desc_pattern.upper()}")
                print(text)
                break
        
        await browser.close()

await run()

FOUND SECTION: DESCRIPTION
The requirements listed in the job descriptions are guidelines. You don’t have to satisfy every requirement or meet every qualification listed. If your skills are transferable we would still love to hear from you.


In [5]:
low_quality_urls = ['https://jobs.lever.co/binance/?location=Australia%2C%20Brisbane', 'https://corporate.deeplearning.ai/courses/practical-multi-ai-agents-and-advanced-use-cases-with-crewai/lesson/aix1c/content-creation-at-scale', 'https://jobs.lever.co/atmosera/c5e21ae4-fcf3-4fe0-83df-a018a4f9f1b1', 'https://apply.workable.com/netguru/j/60648FBA5C/', 'https://github.com/TEN-framework/TEN-Agent', 'https://fastgpt.io/en', 'https://redirect.github.com/speedyapply/2026-AI-College-Jobs', 'https://github.com/ChatFAQ/ChatFAQ', 'https://www.qcloud.com/developer/article/2628717', 'https://jobs.lever.co/binance?department=Engineering', 'https://apply.workable.com/mlabs/j/CF4E6EC215/', 'https://dify.ai/', 'https://jobs.sequoiacap.com/jobs/hugging-face', 'https://jobs.ashbyhq.com/Pyyne/11176bae-e8bf-4163-9f07-949941f80c84', 'https://dailyaimail.news/news/career-ops-claude-code-job-search-open-source', 'https://apply.workable.com/tiger-analytics/j/39C8151486/', 'https://unsloth.ai/docs/models/qwen3.5/fine-tune', 'https://apply.workable.com/huggingface/', 'https://www.useparallel.com/app/candidate/job/674e4ffe79c5f73418602259', 'https://www.digit.in/features/general/he-used-claude-code-to-apply-to-over-700-jobs-and-got-hired-heres-how.html', 'https://www.wantedly.com/projects/2275794', 'https://jobs.lever.co/binance/fe2b0bf9-95bb-40c5-959b-f991278a1cbe', 'https://github.com/primeIntellect-ai/prime-rl', 'https://www.facebook.com/groups/573518645176483/posts/809888104872868/', 'https://jobs.lever.co/binance?%20Design=&department=Engineering', 'https://apply.workable.com/volga-partners/j/35FCA61135', 'https://jobs.ashbyhq.com/cohere/1fa01a03-9253-4f62-8f10-0fe368b38cb9']

### page.content() + BS4 <<< GO WITH THIS ONE 

In [ ]:
invalid_jobs = [
    Job(
        title='Hugging Face - Current Openings', 
        url='https://apply.workable.com/huggingface/', 
        text='Hugging Face - Current Openings', 
        searched_via='serp', 
        low_quality=True
    ), 
    
    Job(title='AI Engineer - AI/LLM (Backend) - Volga Partners', url='https://apply.workable.com/volga-partners/j/35FCA61135', text='AI Engineer - AI/LLM (Backend) - Volga Partners', searched_via='serp', low_quality=True), Job(title='Junior AI Engineer - Netguru', url='https://apply.workable.com/netguru/j/60648FBA5C/', text='Junior AI Engineer - Netguru', searched_via='serp', low_quality=True), Job(title='AI Data Engineer - MLabs', url='https://apply.workable.com/mlabs/j/CF4E6EC215/', text='AI Data Engineer - MLabs', searched_via='serp', low_quality=True), Job(title='Binance', url='https://jobs.lever.co/binance?%20Design=&department=Engineering', text=None, searched_via='serp', low_quality=True), Job(title='Binance', url='https://jobs.lever.co/binance?department=Engineering', text=None, searched_via='serp', low_quality=True), Job(title='Atmosera - Collaborative Engineer, GitHub Copilot (Remote', url='https://jobs.lever.co/atmosera/c5e21ae4-fcf3-4fe0-83df-a018a4f9f1b1', text=None, searched_via='serp', low_quality=True), Job(title='Binance', url='https://jobs.lever.co/binance/?location=Australia%2C%20Brisbane', text=None, searched_via='serp', low_quality=True), Job(title='Analytics Consulting(Telecom/Media/Entertainment) - Tiger ...', url='https://apply.workable.com/tiger-analytics/j/39C8151486/', text='Associate Director - Analytics Consulting(Data Science) - Tiger Analytics Inc.', searched_via='serp', low_quality=True), Job(title='AI Automation Engineer @ Pyyne', url='https://jobs.ashbyhq.com/Pyyne/11176bae-e8bf-4163-9f07-949941f80c84', text='AI Automation Engineer @ Pyyne\n\nYou need to enable JavaScript to run this app.', searched_via='serp', low_quality=True), Job(title='Binance - Software Engineer (AI/LLM Experience a Plus)', url='https://jobs.lever.co/binance/fe2b0bf9-95bb-40c5-959b-f991278a1cbe', text=None, searched_via='serp', low_quality=True), Job(title='Applied AI Engineer – Agentic Workflows @ Cohere', url='https://jobs.ashbyhq.com/cohere/1fa01a03-9253-4f62-8f10-0fe368b38cb9', text='Applied AI Engineer – Agentic Workflows @ Cohere\n\nApplied AI Engineer – Agentic Workflows @ Cohere', searched_via='exa', low_quality=True), Job(title='Dify', url='https://dify.ai/', text="Dify is a Technology, Information and Internet company. Dify.AI is an open-source llm app development platform that orchestrates llm apps from agents to complex workflows with an rag engine. It offers a thriving marketplace for generative AI applications. Dify employs 47 people (+126.8% YoY), founded in 2023. Headquartered in Sunnyvale, California, United States, operates in 9 countries (including China, Japan, United Kingdom, India, and Hong Kong). Has CNY20.0M in total funding, most recently a LangGenius - Series A in August 2024, with 2 prior funding rounds. ## About Dify offers everything you need — agentic workflows, RAG pipelines, integrations, and observability — all in one place, putting AI power into your hands. ## Company Details\n- Industry: Technology, Information and Internet\n- Type: Partnership - Headquarters: Sunnyvale, United States\n- Founded Year: 2023\n- Homepage: dify.ai\n- Status: active   ...    ## Workforce\n- Employees: 47\n - Monthly Growth: +13.4%\n - Yearly Growth: +126.8%\n- Key Executives:\n - Zheng Wang: CMO   ...    - By Country: United States: 16 (17%), China: 15 (16%), Japan: 5 (5%), United Kingdom: 4 (4%), India: 2 (2%), Hong   ...    - By Seniority: Specialist: 27 (29%), Other Management: 11 (12%), Manager: 6 (6%), C-Level: 2 (2%), Head: 1 (1%),   ...    - Sandbox Plan (Free)\n- Professional ($59.00): 1 Month\n- Team ($159.00): 1 Month   ...    - Top Countries: China: 58%, Japan: 10%, United States: 9%, Singapore: 4%, Hong Kong: 2% ## Categories technology/software development, generative ai software, large language model operationalization (llmops) software, dify.ai, generative ai infrastructure, other development, large language model operationalization (llmops), agentic ai, developer tools, generative ai, saas, workflow, ai solutions, agent, genai, llm, rag, llmops, generative ai   ...    ## Locations (2 offices across 1 country)\n- United States (2 offices)\n - Sunnyvale, CA 94085 (HQ)   ...    - [2025-10-02] ニフティニュース: （プレスリリース）D-Marketing Academy、生成AI講座数300超のeラーニングに、ノーコードAIアプリ開発ツール「Dify」「n8n」講座を大幅拡充   ...    - [2026-02-12] Many LLM features don’t fail because the model is “bad.” They fail because the prompts aren’t designed   ...    - [2026-02-10] If you’ve ever shipped an LLM feature and wondered “𝙒𝙝𝙮 𝙙𝙤𝙚𝙨 𝙞𝙩 𝙬𝙤𝙧𝙠 𝙥𝙚𝙧𝙛𝙚𝙘𝙩𝙡𝙮 𝙨𝙤𝙢𝙚𝙩𝙞𝙢𝙚𝙨 — 𝙖𝙣𝙙   ...    - [2026-01-27] On January 21, Our first Dify Studio Taiwan Meetup at Zeabur came to a close. We’d like to sincerely   ...    - [2026-01-23] I’m attending ADSW representing LangGenius, K.K. Dify now has direct support in the Middle East. Dify is a top-tier, low-code LLM platform to quickly build RAG knowledge systems and AI workflows, w... (11 reactions) - [2026-01-22] New to Dify? This walkthrough breaks down what it does, where it fits, and when to use it. If you're tired of rebuilding prompts, RAG pipelines, and tool integrations from scratch, this one's for y... (5 reactions, 1   ...    - [2026-01-20] Buried in resumes this week? Join our Dify 101 webinar and learn how to build two Resume Screening   ...    - [2026-01-16] Great energy at the Taiwan Artificial Intelligence Association event—Alvin Wu from our team shared how Dify powers enterprise Agentic Workflows, from RAG", searched_via='exa', low_quality=True), Job(title='He used Claude Code to apply to over 700 jobs and got hired: Here’s how', url='https://www.digit.in/features/general/he-used-claude-code-to-apply-to-over-700-jobs-and-got-hired-heres-how.html', text="He used Claude Code to apply to over 700 jobs and got hired: Here’s how # He used Claude Code to apply to over 700 jobs and got hired: Here’s how\n\nAdd DIGIT as a preferred source   ...    When I was searching for a job right out of college, I had a spreadsheet to track all my applications. While that is a workable method, one developer has leapfrogged it with an AI pipeline instead. Santiago, a former founder and now Head of Applied AI, spent his job search engineering his way out of the search problem. Manually sifting through listings and tailoring CVs one by one is a pretty time consuming task so he built Career-Ops, an AI-powered job search system on top of Claude Code. He used it to evaluate over 740 job offers, generate more than 100 tailored CVs, and ultimately   ...    bro created an AI job search system for Claude Code that scored 700+ job applications and actually got him a job. AND IT'S NOW OPEN-SOURCE.It scans multiple company career pages, rewrites your CV per job, and even fills application   ...    Also read: Microsoft Copilot terms are legal ‘LOL’ for every business using it\n\n## The system he built Career-Ops runs off a single slash command inside Claude Code. Paste in a job URL or description, and it takes over – scraping the listing, scoring the opportunity, generating a tailored CV, and logging everything to a tracker   ...    The scoring engine evaluates each role across ten weighted dimensions, assigning a grade from A to F. The system is   ...    Also read: Claude AI has functional emotions that influence behaviour, Anthropic study finds The repo ships with 14 skill modes covering the full job search lifecycle. A portal scanner comes pre-configured with 45+ companies – from Anthropic and OpenAI to ElevenLabs and n8n – searching across Ashby, Greenhouse, Lever, and Wellfound simultaneously. When it finds a match, it generates an ATS-optimized PDF using Playwright, with keyword injection and custom typography baked in. There is also a batch mode for processing ten or more offers in parallel using Claude sub-agents, a form-filling mode for live applications, and a LinkedIn outreach generator. A terminal dashboard built in Go lets you browse, filter, and sort your entire pipeline without leaving the command line.   ...    What makes Career-Ops genuinely interesting is not just what it automates, but how it is designed. The system reads and edits its own configuration files, meaning you can ask Claude to change scoring weights, swap archetypes, or add companies to the scanner and it does it. The AI is not a feature bolted on. It is the interface. The repo is live on GitHub", searched_via='exa', low_quality=True), Job(title='Open-Source Claude Code Job Search System Goes Viral After Helping Its Creator Get Hired | Daily AI Mail', url='https://dailyaimail.news/news/career-ops-claude-code-job-search-open-source', text="Open-Source Claude Code Job Search System Goes Viral After Helping Its Creator Get Hired | Daily AI Mail The latest viral Claude Code project is not another coding agent demo. It is a job-search engine that turns AI into a full application pipeline, and its creator says it did more than save time: it helped him actually get hired. The project, called Career-Ops, was highlighted in a post from @Hesamation on X and quickly caught attention because the pitch is unusually concrete. According to the repo and the social post, the system evaluated more than 700 job opportunities, generated over 100 tailored CVs, and helped its builder land a Head of Applied AI role. bro created an AI job search system for Claude Code that scored 700+ job applications and actually got him a job. AND IT'S NOW OPEN-SOURCE.It scans multiple company career pages, rewrites your CV per job, and even fills application forms. The repo has:> 14 skill modes… pic.twitter.com/VOM4M5jzU6   ...    What makes Career-Ops stand out is that it is not framed as a mass-application spam machine. In the project’s own documentation, the emphasis is on scoring fit before applying, then tailoring materials to the role rather than   ...    That approach gives the tool a more serious edge than the usual “AI applies to jobs for you” gimmick. Career-Ops scans company career pages, evaluates openings across weighted dimensions, rewrites resumes for specific listings, generates ATS-friendly PDFs through Playwright, and tracks the pipeline inside a terminal dashboard. The repo also comes with more than 45 preconfigured companies, including Anthropic, OpenAI, ElevenLabs, and Stripe.   ...    The timing is part of the appeal. Claude Code has become one of the hottest developer tools in AI, and builders are increasingly using it as a programmable work engine rather than a chatbot. Career-Ops pushes that idea into a category   ...    There is also a credibility factor here that many viral repos lack. The builder behind the project, santifer, is not claiming abstract productivity gains. The claim is that the system handled a real search, processed real opportunities, and contributed to a real hiring outcome. That makes the repo feel less like a concept demo and more like a proof point for where agent-style tooling is going next. For developers, operators, and anyone testing how far Claude Code can stretch beyond software tasks, Career-Ops is a sharp example of the shift already underway. AI tools are", searched_via='exa', low_quality=True), Job(title="Some remote AI, data, and language roles I'm seeing hiring ...", url='https://www.facebook.com/groups/573518645176483/posts/809888104872868/', text='AI WORK HIVE - OUTLIER, ALIGNERR, DATAANNOTATION, STELLAR, INVISIBLE ETC. | Some remote AI, data, and language roles I’m   ...    AI WORK HIVE - OUTLIER, ALIGNERR, DATAANNOTATION, STELLAR, INVISIBLE ETC. | Some remote AI, data, and language roles   ...    AI WORK HIVE - OUTLIER, ALIGNERR, DATAANNOTATION, STELLAR, INVISIBLE ETC. ·\nJoin\nRobert Norinder · 4d · Shared with Public group\nSome remote AI, data, and language roles I’m seeing hiring right now I track remote AI jobs each week and send out a curated list. Sharing a shortened version here in case it’s useful. To keep this readable, I’m limiting it to a maximum of 2 roles per category. These are the ones that stood out most   ...    All links go directly to company hiring pages.\nPay is listed where available. These all appear legitimately open right now.\nAI Training & Evaluation\nData Annotation Contributor — Human Signal Remote — $20/hr Dataset labeling and evaluation work supporting large-scale AI training pipelines. This sits closer to the   ...    https://job-boards.greenhouse.io/humansignal/jobs/5823725004\nAI Trainer (Part-time) — Kastle   ...    Structured evaluation and feedback work improving how AI systems perform across defined workflows. Part-time with   ...    https://jobs.ashbyhq.com/.../824fc6f9-b2ff-44ea-bc61...\nAnnotation & Data Work   ...    https://apply.workable.com/j/4086A366AA\nAudio & Speech Data   ...    https://app.opentrain.ai/job.../cmmsnoabk000n04l43jq3gaa7 Language Data Quality Reviewer (Korean) — Volga Partners\nRemote — $5–$10/hr Transcription and quality review work supporting Korean-language datasets used in speech and language models. https://apply.workable.com/volga-partners/j/447505F2F0/\nTechnical & AI Builder Roles AI Engineer — Mind Computing\nRemote (US) — $115k–$125k Full-time role building and deploying AI systems rather than evaluating them. Fewer of these roles show up compared to   ...    https://mind-computing.breezy.hr/.../edde74233ea2-ai...\nAI Engineer (Intern) — Juniper Square Remote (US and Canada) — \\~$50/hr High-paying internship working on production systems inside a real company environment. Likely more hands-on than   ...    https://jobs.ashbyhq.com/.../48d40321-aba6-46a8-a438...\nAI-Adjacent Roles\nAI Analyst — BH Remote — $85,000–$95,000 Role focused on turning AI outputs into usable insights inside business workflows. This is where a lot of real-world   ...    https://www.paycomonline.net/.../C8CA71BADF7E.../jobs/575059   ...    The pattern is still the same: most hiring is not for building models, but for improving them. Evaluation, annotation, language work, and applied roles are where a lot of the demand is. If you want the full weekly list (15 roles), I send it out here:\nhttps://jobsignal.work If you want to browse more broadly, there are currently 50,000+ remote roles listed here: [[https://all', searched_via='exa', low_quality=True), Job(title='Spring AI系列之RAG（检索增强生成）从原理到实战指南-腾讯云开发者社区-腾讯云', url='https://www.qcloud.com/developer/article/2628717', text="Spring AI系列之RAG（检索增强生成）从原理到实战指南-腾讯云开发者社区-腾讯云\n\n## Spring AI系列之RAG（检索增强生成）从原理到实战指南\n\n原创\n\n关注作者\n\n腾讯云 开发者社区\n\n文档 建议反馈 控制台\n\n登录/注册\n\n首页\n\n学习\n\n活动\n\n专区\n\n圈层\n\n工具   ...    社区首页> 专栏>Spring AI系列之RAG（检索增强生成）从原理到实战指南\n\n# Spring AI系列之RAG（检索增强生成）从原理到实战指南\n\nSmileNicky\n\n关注 发布 于 2026-02-09 10:23:44\n\n发布 于 2026-02-09 10:23:44\n\n5600\n\n举报\n\n概述 在LLM（大语言模型）时代，如何让AI既拥有通用能力又具备专业知识？RAG技术给出了完美答案。本文将基于Spring AI生态，深入剖析RAG的核心原理、实现细节与优化策略。 文章被收录于专栏： Nicky's blog Nicky's blog\n\n \n\n原创声明：本文系作者授权腾讯云开发者社区发表，未经许可，不得转载。 如有侵权，请联系 cloudcommunity@tencent.com删除。\n\n媒体 AI\n\n原创声明：本文系作者授权腾讯云开发者社区发表，未经许可，不得转载。 如有侵权，请联系 cloudcommunity@tencent.com删除。\n\n媒体 AI\n\n评论\n\n登录后参与评论\n\n0 条评论\n\n热度\n\n最新\n\n登录 后参与评论\n\n推荐阅读   ...    云数据库为企业提供了完善的关系型数据库、非关系型数据库、分析型数据库和数据库生态工具。您可以通过产品选择和组合搭建，轻松实现高可靠、高可用性、高性能等数据库需求。云数据库服务也可大幅减少您的运维工作量，更专注于业务发展，让企业一站式享受数   ...    产品介绍\n\n2026采购季 | AI焕新·智启新局\n\n领券\n\n### 社区 - 技术文章\n- 技术问答\n- 技术沙龙\n- 技术视频\n- 学习中心\n- 技术百科   ...    ### 活动\n\n- 自媒体同步曝光计划\n- 邀请作者入驻\n- 自荐上首页\n- 技术竞赛\n\n### 圈层 - 腾讯云最具价值专家\n- 腾讯云架构师技术同盟\n- 腾讯云创作之星\n- 腾讯云TDP\n\n### 关于 - 社区规范\n- 免责声明\n- 联系我们\n- 友情链接\n- MCP广场开源版权声明\n\n### 腾讯云开发者   ...    - 域名注册\n- 云服务器\n- 区块链服务\n- 消息队列\n- 网络加速\n- 云数据库   ...    - 人脸识别\n- 腾讯会议\n- 企业云\n- CDN加速\n- 视频通话\n- 图像分析   ...    - 数据安全\n- 负载均衡\n- 短信\n- 文字识别\n- 云点播\n- 大数据 - 小程序开发\n- 网站监控\n- 数据迁移 Copyright © 2013 - 2026 Tencent Cloud. All Rights Reserved. 腾讯云 版权所有 深圳市腾讯计算机系统有限公司 ICP备案/许可证号： 粤B2-20090059 粤公网安备44030502008569号 腾讯云计算（北京）有限责任公司京ICP证150476号 | [京ICP备110187", searched_via='exa', low_quality=True), Job(title='Practical Multi AI Agents and Advanced Use Cases with crewAI - DeepLearning.AI', url='https://corporate.deeplearning.ai/courses/practical-multi-ai-agents-and-advanced-use-cases-with-crewai/lesson/aix1c/content-creation-at-scale', text="Practical Multi AI Agents and Advanced Use Cases with crewAI - DeepLearning.AI This is another incredible use case. For this, we're going to be building a crew that is able to create content   ...    posts check out. Now let's go back into the code and keep on building. Now let's import our CrewAI tools that we're going to be using for this. The tools are pretty simple. We're using our usual serperdev tool that allows us to do   ...    new tool into this, and that is the website's search tool. This is a RAG as a tool. What it does is, as our agents   ...    will generate embeddings and they will choose what to search and when to search it. Now that we know what the tools we're going to use, let's import and set up or our LLMs that are going to be using for this lesson. Keep in mind that   ...    tasks. So let's start by setting up our OpenAI model. And to do it, we're going to set this open the AI model name environment variable. And we're going to set the value to be GPT-4o-mini. That's already the default for CrewAI. But we're setting it up anyway just to be more verbose and direct. We're also going to use a groq model, specifically Llama 3.1-70b. And we can do that by setting up this variable that we're going to reference later when creating our agents to be the string that represents that model. So it's a groq model. And the model name is Llama 3.1-70b Versatile. Now   ...    post written by these agents for us is starting from one single, very complex subject and then going around and researching the web to acquire a better understanding about the topic, then doing RAG as a tool specific web pages in   ...    amazing content and getting to review this content to make sure that it's rock solid. And here you can have a great use case. And we have been seeing some customers and user of CrewAI, actually deployed this use cases for doing content marketing at a scale. So this unlocks a lot of use cases and how you could use this not only to produce content, but using these different tools and these different models in order to build interesting things yourself. Remember that two of these four agents we're running Llama 70b using groq. And that's super fast. So you can have a few agents are going to be optimized for speed. They're going to be using models like this. And providers like groq and other agents are going to be using their models like in the case of our content writer agents that were using ChatGPT. This was a very interesting lesson, and I hope you're having a lot of fun, at least as much fun as I am. Now let's jump in the next one and start to talk about more use cases. Okay. I'm going to see you then in a few seconds. course detail\n\nSign in to continue learning # Practical Multi AI Agents and Advanced Use Cases with crewAI\n\nBeginner\n\n2 hours 39 mins\n\nJoin Now\n\n### Topics Agents\n\nChatbots\n\nGenAI Applications\n\nGenerative Models\n\nTask Automation\n\n### Collaborator\n\ncrewAI Practical Multi AI Agents and Advanced Use Cases with crewAI - IntroductionVideo・4 mins\n- Overview of Multi AI-Agent SystemsVideo・12 mins - Automated Project: Planning, Estimation, and AllocationVideo with Code Example・14 mins - Internal and External IntegrationsVideo・3 mins - Building Project Progress ReportVideo with Code Example・12 mins - Complex crew SetupsVideo・3 mins\n- Agentic Sales PipelineVideo with Code Example・28 mins - Performance OptimizationVideo・7 mins - Support Data Insight AnalysisVideo with Code Example・25 mins - Multi-Model Use CasesVideo・3 mins\n- Content Creation at ScaleVideo with Code Example・17 mins - Agentic Workflows in IndustryVideo・10 mins\n- [Generate, Deploy and Monitor", searched_via='exa', low_quality=True), Job(title='GitHub - speedyapply/2026-AI-College-Jobs: 2026 AI/ML internship & new graduate job list updated daily · GitHub', url='https://redirect.github.com/speedyapply/2026-AI-College-Jobs', text='GitHub - speedyapply/2026-AI-College-Jobs: 2026 AI/ML internship & new graduate job list updated daily · GitHub / 2026-AI-College-Jobs Public\n\nStar 5k - Pricing\n- Notifications\n- Fork 199\n\n \n\n main Branches Tags\n\n \n\nGo to file\n\nCode\n\nLast commit message\n\nLast commit date 580 Commits\n\n.github\n\n.github\n\nINTERN_INTL.md INTERN_INTL.md\n\nNEW_GRAD_INTL.md\n\nNEW_GRAD_INTL.md\n\nNEW_GRAD_USA.md   ...    # 2026 Artificial Intelligence Internship & New Grad Positions This repository is a comprehensive list of AI/ML & Data Science jobs for college students in search of internships or new graduate positions. The positions are updated daily, and we prioritize jobs posted within the last 120 days. ### USA Positions 🦅 - Internships 📚- 351 available (FAANG+, Quant, Other) - New Graduate 🎓- 234 available (FAANG+, Quant, Other) ### International Positions 🌐 - Internships 📚- 329 available (FAANG+, Quant, Other) - New Graduate 🎓- 273 available (FAANG+, Quant, Other)   ...    ## 2026 USA AI Internships 📚🦅\n\n🔽End of List\n\n### FAANG+ | Company | Position | Location | Salary | Posting | Age |\n| --- | --- | --- | --- | --- | --- | | Adobe | 2026 Intern - Machine Learning Engineer | San Jose | $55/hr | 1d | | Roblox | 2026 Machine Learning Engineer - Recommendation Systems - Intern | San Mateo, CA, United States | $64/hr | 6d | | Meta | Research Scientist Intern - Applied Vision in Augmented Reality - PhD | Redmond, WA | $50/hr | 13d   ...    | Company | Position | Location | Posting | Age |\n| --- | --- | --- | --- | --- |   ...    | Bosch | AI Systems Engineering Intern | Sunnyvale, CA, us | 0d | | ITC Federal | Summer Intern - AI/LLM Development - Growth Solutions Architecture Team | US VA Fairfax |   ...    | Planet | Intern - AI/ML | Remote | 0d |   ...    | Bosch | LLM & Agentic AI R&D Intern | Sunnyvale, CA, us | 2d | | Cloudforce | AI Enablement Intern | National Harbor, Maryland | 2d | | Cloudforce | AI Agent Builder Intern | National Harbor, Maryland | 2d |   ...    | Micron Technology | Intern - EHSS Agentic AI & Multi Agent Orchestration - Enterprise Workflows | Boise,   ...    | Proofpoint | Agentic AI Workflows Intern | Sunnyvale, CA | 9d |  | Pacvue | AI Accelerator   ...    | Veolia | AI Engineering Intern - LLM | Paramus, NJ, us | 23d |   ...    | HRL Laboratories | LLM & Agentic AI Research Intern | Calabasas, CA | 28d |  | Cadence | AI &   ...    | Zillow | AI Applied Scientist - PhD Intern - Next-Gen Agent', searched_via='exa', low_quality=True), Job(title='ChatFAQ/ChatFAQ', url='https://github.com/ChatFAQ/ChatFAQ', text='# Repository: ChatFAQ/ChatFAQ Open-source ecosystem for building AI-powered conversational solutions using RAG, agents, FSMs, and LLMs.   ...    - Topics: ai, answer-generation, chatbots, chatgpt, claude, generative-ai, gpt4, information-retrieval, intent-detection, large-language-models, llm, nlg, open-source, parsing, rag, retrieval-augmented-generation,   ...    - Default branch: develop\n- Homepage: https://www.chatfaq.io/\n- Created: 2023-03-06T14:01:29Z   ...    ---\n\n## Group 403 (1) - An Open Source RAG & Agent ecosystem for your business needs **ChatFAQ** is an open-source platform and framework for creating diverse AI-powered conversational solutions: - LLM-based chatbots\n- RAG-enhanced chatbots\n- Agentic workflows - Rule-based Finite State Machines with LLM assistance\n- Hybrid solutions combining multiple approaches **ChatFAQ** fully relies on **open-source technologies**, allowing for flexibility, privacy, full control and costs-effectiveness. It includes a SDK **to build your specialized NLG engine** and customized chat widgets,   ...    https://github.com/ChatFAQ/ChatFAQ/assets/127191313/7927f51f-d7ac-40e5-b4d0-62081742de4f\n\n### Documentation The official documentation is hosted on Read the Docs.\n\n### Components\n\nChatFAQ Components - **ChatFAQ SDK**: A SDK to build agents, RAG pipelines, Finite State Machines and any other AI flow you can imagine. - **Chat Widget**: Embed a customizable chat interface into your website. - **Admin Dashboard**: A dashboard to manage all your knowledge bases, LLMs, retrievers, label conversations, see   ...    - **Ray Cluster**: Power indexing pipelines, LLM inference, retrieval operations and more. - **Backend**: Django-based system to orchestrate all components.\n- **Foundational Models**: - Primary: vLLM integration for open-source LLMs - Additional: Integrations with OpenAI, Anthropic, Mistral, Gemini and Together. ### Visit Us!\n\nFor more information about ChatFAQ and any additional needs, feel free to visit our website Or chat with us on Discord for any requests or inquiries about this repository. \n\n \n \n \n\n### Examples Here are some examples of how to use the ChatFAQ SDK to build different types of conversational solutions: - **LLM Example**: This example demonstrates a simple chatbot using an LLM to generate responses.   ...    - **R', searched_via='exa', low_quality=True), Job(title='Hugging face | ML Research Engineer Internship, Agent AI', url='https://www.useparallel.com/app/candidate/job/674e4ffe79c5f73418602259', text='Hugging face | ML Research Engineer Internship, Agent AI - EMEA Remote\n\nLoading...', searched_via='exa', low_quality=True), Job(title='Jobs at Hugging Face | Sequoia Capital', url='https://jobs.sequoiacap.com/jobs/hugging-face', text='Jobs at Hugging Face | Sequoia Capital OUR FOUNDERS OUR COMPANIES OUR TEAM COMPANY DESIGN STORIES Our Founders Our Companies Our team Company Design stories ABOUT Our Ethos Our History FAQ Jobs Newsletter Legal BUSINESS ENTITIES U.S./Europe India Southest Asia China Heritage SCGE LOGIN LP Login Sequoia Ampersand Login\n\n© 2021 Sequoia', searched_via='exa', low_quality=True), Job(title='AIで業務変革を実装するLLM基盤コンサルタント募集！ - 株式会社ログラスのWeb Engineerの採用 - Wantedly', url='https://www.wantedly.com/projects/2275794', text='AIで業務変革を実装するLLM基盤コンサルタント募集！ - 株式会社ログラスのWeb Engineerの採用 - Wantedly\n\n400万人が利用するビジネスSNS\n\nアプリを使う - LLM基盤コンサルタント\n- 6エントリー\n\n# AIで業務変革を実装するLLM基盤コンサルタント募集！\n\n## 株式会社ログラス\n\nLLM基盤コンサルタント\n\nMid-career\n\n6エントリー\n\non 2025/12/01 205 views\n\n6人がエントリー中\n\n# AIで業務変革を実装するLLM基盤コンサルタント募集！\n\n## 株式会社ログラス\n\nTokyo\n\nMid-career\n\nOnline interviews OK\n\nTokyo   ...    【企業概要】 株式会社ログラスは「良い景気を作ろう。」をミッションに、企業経営・予実管理領域のDXと高度化を目指し、「Loglass 経営管理」を中心に、「Loglass 人員計画」「Loglass 販売計画」「Loglass   ...    あらゆる企業の経営における意思決定をよりよいものにし、顧客企業の業績向上と社会全体の良い景気の実現にチャレンジしています。 ※プロダクトビジョン詳細はぜひこちらのnoteをご覧ください。 https://note.com/loglass_sakamoto/n/n2aaea5974077 ◻︎募集ポジションについて（チームのビジョン・体制・募集背景 ） 私たちは、「AIを業務に溶け込ませる」世界の実現に向け、LLM・AIエージェントを企業の中核業務へ本格実装する取り組みを推進しています。 近年、企業のAI活用はPoC（実証）から“本番活用”へと急速にシフトしています。 しかし、実際の大規模業務の中では、 - どの業務にAIを適用すべきか？ - どのようなワークフローが最適なのか？ - どこまでLLMが担えるのか？ - 業務要件を踏まえてどう設計すれば動くのか？ といった 「技術と業務の翻訳」が極めて重要になります。 そこで、顧客の業務を深く理解し、AIエージェントのユースケースを設計し、技術チームと共に“実装”まで導くコンサルタントを新たに募集します。 ◻︎チームについて 所属予定チーム：新規事業部、LLM基盤チーム - CEO直下のAI・LLM領域のクロスファンクショナルチーム - コンサル、FDE（エンジニア）、PdM、デザイナーが密に連携し、課題発見から実装までを一気通貫で担当 - エンタープライズ企業の経営企画・業務部門と直接連携し、AI導入の成功パターンを創出 - LLM基盤チームと協働し、ユースケースの共通化・AIエージェントの標準化も推進 「机上の戦略」ではなく、実際に動くAIエージェントを顧客とつくる」 そんな“実装特化型”のコンサル組織です。 ◻︎具体的な業務内容・ミッション ＜業務内容＞ - 顧客の業務プロセス分析・課題の構造化 - AI適用ポイントの特定とユースケース設計 - LLM／AIエージェントの要件定義（FDEと連携） - 業務フロー・ナレッジ・ツール連携を踏まえたAIエージェント設計 - 導入プロジェクトの推進（要件整理〜導入〜定着まで） - AI精度評価・改善サイクルの運用 - 顧客で生まれた成功パターンの標準化・プロダクトへのフィードバック - ステークホルダー管理、進捗・リスクのハンドリング ＜ミッション＞ - LLM技術を「使える形」で事業価値に転換する - 高速に検証→改善→実装できるAI基盤の構築 - 複雑な文書業務をAIで処理できるドメイン特化基盤を作り上げる - 顧客企業での“AI活用の成功例”を再現性ある形で創出する ◻︎要件 ＜MUST＞ - コンサルティングファームでの実務経験（3年以上） - 顧客課題の構造化／プロジェクトマネジメント経験 - エンジニアを巻き込んだ要件定義・実装ディレクション経験 - 日本語でのビジネスコミュニケーション力 - 以下いずれかの経験 - AI／データ活用プロジェクトの実務経験 - プリセールス／セールスエンジニア経験 - エンジニアリング経験 ＜WANT＞ - エンタープライズ企業向けの業務改革・DXプロジェクト経験 - プロダクト企画（顧客要件→機能要件変換）の経験 - データサイエンス／機械学習／LLMに関する知見 - AIエージェントやAIワークフローの理解 - コンサル×プロダクト開発の両面に興味がある方 ＜求める人物像＞ - AI/LLMの社会実装・業務変革に強い関心がある - 顧客現場に深く入り込み、課題を自分ごととして扱える -   ...    0人がこの募集を応援しています\n\n応援する\n\n続きを読む\n\n続きを読む\n\n### ログラス開発チームの雰囲気、取り組みを紹介！ ~開発チームの1日~\n\nRecruiting Loglass   ...    ### 話を聞き', searched_via='exa', low_quality=True), Job(title='FastGPT - Enterprise AI Agent Builder | Open Source RAG Platform', url='https://fastgpt.io/en', text='FastGPT - Enterprise AI Agent Builder | Open Source RAG Platform\n\n50w+ Users built specialized AI Agents with FastGPT 27.4 k\n\n# FastGPT - Enterprise AI Agent Builder Flexible AI Workflow + AI Knowledge Base + Template System + Agentic RAG = Powerful AI Agent Builder Contact Sales\n\nGet started\n\n##### Video Presentation\n\n#### Features ## Domain-Specific AI Assistant Create AI-powered chatbots for specific domains by training models with imported documents or Q&A pairs. ## Automated Data Preprocessing Save time and improve efficiency with automated text preprocessing, vectorization, and QA segmentation. ## Workflow Orchestration Support AI Workflow orchestration, Design complex workflow using a visual drag-and-drop interface, integrating tasks   ...    ## Seamless API Integration Seamlessly connect with existing GPT applications and platforms like Discord, Slack, and Telegram using OpenAI-aligned   ...    #### Why FastGPT?\n\nDiscover the advantages of FastGPT\n\n## Open Source\n\nSecure and reliable open-source codebase. ## Optimized Q&A\n\nEnhanced question-answering accuracy for customer service.\n\n## Visual Workflow Design complex workflows with ease using the Flow module.\n\n## Seamless Extensibility Seamlessly integrate FastGPT into your applications via API.\n\n## Debugging Tools Refine your models with comprehensive debugging features.\n\n## Multi-Model Compatibility Compatible with various LLM models, with more to come. Like this open-source AI Agent Builder? Star it on GitHub! 🌟\n\n#### Frequently Asked Questions Frequently asked questions about FastGPT\n\nView more\n\n## Can the open-source version be used commercially? FastGPT allows commercial usage, such as serving as a backend service for other applications or as an application development platform for enterprises. However, when it comes to multi-tenant SaaS services or matters involving the LOGO and copyright information, you must contact the author to obtain a commercial license. ## What document formats can be imported into the FastGPT knowledge base? FastGPT supports importing documents in various formats, including Word, PDF, Excel, Markdown, and web links. It also enables syncing data from an entire website, automatically handling text preprocessing, vectorization, and QA splitting, which saves manual training time and improves efficiency. ## Which models can be integrated with FastGPT? As long as the API of the model you want to integrate aligns with the official OpenAI API, it can be used with FastGPT. You can utilize projects like One API to unify access to different models and provide an API that is compatible with   ...    If you', searched_via='exa', low_quality=True), Job(title='TEN-framework/ten-framework', url='https://github.com/TEN-framework/TEN-Agent', text="# Repository: TEN-framework/ten-framework\n\nOpen-source framework for conversational voice AI agents - Stars: 10378\n- Forks: 1243\n- Watchers: 10378\n- Open issues: 190\n- Primary language: Python - Languages: Python (31.5%), C (22.6%), C++ (15.6%), TypeScript (13.1%), Rust (11.7%), Go (3.1%), Shell (0.9%),   ...    - License: Other (NOASSERTION)\n- Topics: ai, multi-modal, real-time, video, voice\n- Default branch: main - Homepage: https://agent.theten.ai/\n- Created: 2024-06-19T14:26:15Z\n- Last push: 2026-03-29T11:49:21Z   ...    [Official Site][official-site] •\n[Documentation][documentation] •\n[Blog][blog]\n\n \n\n \n\n \n Table of Contents - [Welcome to TEN][welcome-to-ten]\n- [Agent Examples][agent-examples-section]   ...    - [Stay Tuned][stay-tuned]\n- [TEN Ecosystem][ten-ecosystem-anchor]\n- [Questions][questions]   ...    ## Welcome to TEN\n\nTEN is an open-source framework for real-time multimodal conversational AI. [TEN Ecosystem][ten-ecosystem-anchor] includes [TEN Framework][ten-framework], [Agent Examples][agent-examples-repo], [VAD][ten-vad], [Turn Detection][ten-turn-detection] and [Portal][ten-portal]. | Community Channel | Purpose |\n| --- | --- | | [![Follow on X][follow-on-x-badge]][follow-on-x] | Follow TEN Framework on X for updates and announcements | | [![Discord TEN Community][discord-badge]][discord-invite] | Join our Discord community to connect with developers | | [![Follow on LinkedIn][linkedin-badge]][linkedin] | Follow TEN Framework on LinkedIn for updates and announcements | | [![Hugging Face Space][hugging-face-badge]][hugging-face] | Join our Hugging Face community to explore our spaces and   ...    ## Agent Examples\n\n \n\n![Image][voice-assistant-image] Multi-Purpose Voice Assistant — This low-latency, high-quality real-time assistant supports both RTC and   ...    [VAD][voice-assistant-vad-example], [Turn Detection][voice-assistant-turn-detection-example], and other extensions.   ...    Doodler — A doodle board that turns spoken or typed prompts into simple hand-drawn sketches, complete with a crayon   ...    LLM-powered communication with hardware.   ...    ## Quick Start with Agent Examples\n\n### Localhost\n\n#### Step ⓵ - Prerequisites   ...    | **Keys** | • Agora [App ID][agora-app-id] and [App Certificate][agora-app-certificate] • [OpenAI][openai-api] API key • [Deepgram][deepgram] ASR • [ElevenLabs][elevenlabs] TTS |   ...    ##### 1. Clone the repo, `cd` into `ai_agents`, and create a `.env` file from `.env.example`   ...    [![][back-to-top]][readme-top]  ## TEN Ecosystem  | Project | Preview |\n| --- | --- | | [**️TEN Framework**][ten-framework-link] Open-source framework for conversational AI Agents.![][ten-framework-shield]   ...    | [**TEN VAD**][ten-vad-link] Low-latency, lightweight and high-performance streaming voice activity detector   ...    Detection enables full-duplex dialogue communication.![][ten-turn-detection-shield] | ![][ten-turn-detection-banner] | | [**TEN Agent Examples**][ten-agent-example-link] Usecases powered by TEN. | ![][ten-agent-example-banner] | | [**TEN Portal**][ten-portal-link] The official site of the TEN Framework with documentation and a   ...    ## Questions TEN Framework is available on these AI-powered Q&A platforms. They can help you find answers quickly and accurately in   ...    | Service | Link |\n| --- | --- |\n| DeepWiki | [![Ask DeepWiki][deepwiki-badge]][deepwiki] |   ...    ## Contributing  We welcome all forms of open-source collaboration! Whether you're fixing bugs, adding features,   ...    ### License  1. The entire TEN framework (except for the folders explicitly listed below) is released pursuant the   ...    [lang-it-readme]: https://github.com/TEN-framework/ten-framework/blob/main/docs/README-IT.md  [official-site]: https://theten.ai   ...    [agent-examples-repo]: https://github.com/TEN-framework/ten-framework/tree/main/ai_agents/agents/examples   ...    [linkedin]: https://", searched_via='exa', low_quality=True), Job(title='Qwen3.5 Fine-tuning Guide | Unsloth Documentation', url='https://unsloth.ai/docs/models/qwen3.5/fine-tune', text='# Qwen3.5 Fine-tuning Guide You can now fine-tune Qwen3.5 model family (0.8B, 2B, 4B, 9B, 27B, 35B‑A3B, 122B‑A10B) with Unsloth. Support includes both vision, text and RL fine-tuning. Qwen3.5‑35B‑A3B - bf16 LoRA works on 74GB VRAM. - Unsloth makes Qwen3.5 train 1.5× faster and uses 50% less VRAM than FA2 setups.   ...    - Fine-tune 0.8B, 2B and 4B bf16 LoRA via our free Google Colab notebooks:   ...    - Full fine-tuning (FFT) works as well. Note it will use 4x more VRAM. - Qwen3.5 is powerful for multilingual fine-tuning as it supports 201 languages. - After fine-tuning, you can export to GGUF (for llama.cpp/Ollama/etc.) or vLLM - Reinforcement Learning (RL) for Qwen3.5 VLM RL also works via Unsloth inference. - We have A100 Colab notebooks for Qwen3.5‑27B and Qwen3.5‑35B‑A3B. If you’re on an older version (or fine-tuning locally), update first:\n\n{% columns %}\n{% column %}\nUnsloth Studio:   ...    {% hint style="warning" %} Please use `transformers v5` for Qwen3.5. Older versions will not work. Unsloth automatically uses transformers v5 by   ...    If training seems slower than usual, it’s because Qwen3.5 use custom Mamba Triton kernels. Compiling those kernels can take longer than normal, especially on T4 GPUs. It is not recommended to do QLoRA (4-bit) training on the Qwen3.5 models, no matter MoE or dense, due to higher than   ...    ### MoE fine-tuning (35B, 122B)\n\nFor MoE models like Qwen3.5‑35B‑A3B / 122B‑A10B / 397B‑A17B: - You can use our Qwen3.5‑35B‑A3B (A100) fine-tuning notebook   ...    - Best to use bf16 setups (e.g. LoRA or full fine-tuning) (MoE QLoRA 4‑bit is not recommended due to BitsandBytes   ...    - Qwen3.5‑122B‑A10B - bf16 LoRA works on 256GB VRAM. If you\'re using multiGPUs, add `device_map = "balanced"` or follow our multiGPU Guide. ### Quickstart\n\n#### 🦥 Unsloth Studio Guide Qwen3.5 can be run and fine-tuned in Unsloth Studio, our new open-source web UI for local AI. With Unsloth   ...    - Train LLMs 2x faster with 70% less VRAM\n- Search, download, run GGUFs and safetensor models - Self-healing tool calling + web search\n- Code execution (Python, Bash)   ...    {% column %}\n\n\n{% endcolumn %}\n{% endcolumns %}\n\n{% stepper %}\n{% step %}\n\n#### Install Unsloth\n\nRun in your terminal:   ...    {% endstep %}\n{% endstepper %}\n\n#### Unsloth Core (code-based) guide: Below is a minimal SFT recipe (works for “text-only” fine-tuning). See also our vision fine-tuning section.   ...    Qwen3.5 is “Causal Language Model with Vision Encoder” (it’s a unified VLM), so ensure you have the usual vision deps   ...    If you\'d like to do GRPO, it works in Unsloth if you disable fast vLLM inference and use Unsloth inference   ...    ```python\nfrom unsloth import FastLanguageModel\nimport torch\nfrom datasets import load_dataset   ...    model, tokenizer = FastLanguageModel.from_pretrained(\n    model_name = "Qwen/Qwen3.5-27B",   ...    Unsloth supports vision fine-tuning for the multimodal Qwen3.5 models. Use the below Qwen3.5 notebooks and   ...    | Qwen3.5- 0.8B | Qwen3.5- 2B | Qwen3.5- 4B | Qwen3.5- 9B |   ...    To fine-tune vision models, we now allow you to select which parts of the mode to finetune. You can select to only   ...    In order to fine-tune or train Qwen3.5 with multi-images, view our multi-image vision guide. ### Reinforcement Learning (RL)\n\nYou can now train Qwen3.5 with RL, GSPO, GRPO etc with our free notebook:   ...    You can run Qwen3.5 RL with Unsloth even though it is not supported by vLLM, by setting `fast_inference=False` when   ...    You can view our specific inference / deployment guides for Unsloth Studio, llama.cpp, vLLM, llama-server, Ollama. #### Save to GGUF\n\nUnsl', searched_via='exa', low_quality=True), Job(title='PrimeIntellect-ai/prime-rl', url='https://github.com/primeIntellect-ai/prime-rl', text="# Repository: PrimeIntellect-ai/prime-rl\n\nAsync RL Training at Scale - Stars: 1244\n- Forks: 249\n- Watchers: 1244\n- Open issues: 86\n- Primary language: Python   ...    ---\n\n \n \n\n \n \n \n \n\n---\n\n \nPRIME-RL: Async RL Training at Scale\n \n\n---\n\n \n \n \n \n \n \n \n \n \n \n \n \n\n## Overview PRIME-RL is a framework for large-scale reinforcement learning. It is designed to be easy to use and hackable, yet capable of scaling to 1000+ GPUs. Here is what we think sets it apart: 1. Fully asynchronous RL for high-throughput agentic training at scale. 2. Performant: built to train 1T+ MoE models on 1000+ GPUs with FSDP2 for training and vLLM for   ...    3. Native integration with `verifiers` environments through the Environments Hub, including built-in support for SWE and agentic environments. 4. End-to-end post-training: SFT, RL training, and evals.\n5. Multi-node deployment with Slurm and Kubernetes support. 6. Multimodal support for VLMs such as Qwen3-VL.\n7. Hackable, modular, and extensible by design.   ...    The trainer works with both Hugging Face and Prime custom `ModelForCausalLM` out of the box. For selected families   ...    | GLM-5 (`glm_moe_dsa`) | `zai-org/GLM-5`, `zai-org/GLM-5-FP8` | yes | ✅ | ❌ | | Qwen3 MoE (`qwen3_moe`) | `Qwen/Qwen3-30B-A3B`, … | yes | ✅ | ✅ |   ...    > *We develop and test on NVIDIA RTX 3090/4090/5090, A100, H100, H200, and B200. If your setup fails, please create an   ...    ### Prerequisites Currently, you **need at least one NVIDIA GPU to use PRIME-RL**. If you don't already have access to one, we recommend our compute platform for everything from renting on-demand single GPUs for developing, debugging and small   ...    We provide end-to-end training examples in the `examples` directory to highlight features of the framework   ...    Follow this guide to learn the basics of Prime-RL. You can train your own models on 1 to 8 GPUs. Ideal for getting   ...    1. **Reverse Text**: Train `Qwen3-0.6B` to reverse a small chunk of text. Demonstrates tiny-scale   ...    3. **Alphabet Sort**: Train `Qwen3-4B-Instruct-2507` to sort names alphabetically. Demonstrates multi-turn RL training via LoRA without SFT warmup. Can be trained on a single H100 GPU in just over an hour. Ideal for exploring   ...    ### Advanced Training: 32 - 2048 GPUs: Follow this guide to train large models on hard reasoning and agentic / swe environments.   ...    1. **Qwen 3 30B - A3B Math**: Train `Qwen3-30B-A3B` to solve hard math problems.   ...    ## Docs\n\nCheck out the docs directory for in-depth guides on how to use PRIME-RL. - **Entrypoints** - Overview of the main components (orchestrator, trainer, inference) and how to run SFT, RL, and evals - **Configs** - Configuration system using TOML files, CLI arguments, and environment variables - **Environments** - Installing and using verifiers environments from the Environments Hub - **Async Training** - Understanding asynchronous off-policy training and step semantics   ...    - **Deployment** - Training deployment on single-GPU, multi-GPU, and multi-node clusters - **Memory Usage** - Techniques for reducing memory usage (activation checkpointing, offloading, EP, CP,   ...    We warmly welcome community contributions! We use issues to track bugs, feature requests, and share our   ...    ```tex\n@misc{primeintellect2025prime-rl,\n  author = {Prime Intellect},\n  title = {PRIME-RL}, url = {https://", searched_via='exa', low_quality=True)
]



In [40]:
from playwright.async_api import async_playwright
import asyncio
from bs4 import BeautifulSoup
from bs4.element import Tag
import time
import random

async def run():
    async with async_playwright() as p:
        await asyncio.sleep(random.randint(1, 3))
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        await page.goto("https://jobs.ashbyhq.com/cohere/1fa01a03-9253-4f62-8f10-0fe368b38cb9")
        # Take some time to wait for the dom to fully load
        await page.wait_for_timeout(3000)

        html = await page.content()

        if html:
            soup = BeautifulSoup(html, "lxml")
            footer = soup.find("footer")

            # Remove footer
            if footer:
                footer.decompose()

            # Remove junk tags
            for tag in soup.find_all(["nav", "aside"]):
                tag.decompose()

            # Remove the other messy HTML elements by classes and IDs
            # Commented this out because this filters out all content sometimes
            
            # keywords = ["footer", "cookie", "privacy", "nav", "menu", "header"]
            # for tag in list(soup.find_all(True)):

            #     if not isinstance(tag, Tag) or tag.attrs is None:
            #         continue
                    
            #     classes = " ".join(tag.get("class", []))
            #     id_ = tag.get("id", "")

            #     combined = f"{classes} {id_}".lower()

            #     if any(keyword in combined for keyword in keywords):
            #         tag.decompose()
            
                
            text = soup.get_text(separator="\n\n", strip=True)

        
        print(text)
        await asyncio.sleep(random.randint(3, 8))
        await browser.close()

await run()

Applied AI Engineer – Agentic Workflows @ Cohere

You need to enable JavaScript to run this app.

Applied AI Engineer – Agentic Workflows

Location

San Francisco; London; Montreal; New York; Toronto

Employment Type

Full time

Location Type

Remote

Department

Modeling

Applied-ML

Who are we?

Our mission is to scale intelligence to serve humanity. We’re training and deploying frontier models for developers and enterprises who are building AI systems to power magical experiences like content generation, semantic search, RAG, and agents. We believe that our work is instrumental to the widespread adoption of AI.

We obsess over what we build. Each one of us is responsible for contributing to increasing the capabilities of our models and the value they drive for our customers. We like to work hard and move fast to do what’s best for our customers.

Cohere is a team of researchers, engineers, designers, and more, who are passionate about their craft. Each person is one of the best in t

## Test aria_snapshot()

Result:
aria_snapshot requires PlayWright version above 1.59 which isn't competible with my current
Python version (3.13). It needs to downgrade my Python to 3.11 and ultimately, aria_snapshot isn't as good or production-ready as page.content() as well.

In [5]:
from playwright.async_api import async_playwright
import asyncio

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        await page.goto("https://webscraper.io/test-sites/e-commerce/allinone")
        
        snapshot = await page.aria_snapshot(mode="ai")

        print(snapshot)

        await browser.close()
        

await run()

AttributeError: 'Page' object has no attribute 'aria_snapshot'